# 1 - Data Acquisition

This first part gathers the nine benchmark datasets from the 10th DIMACS Implementation Challenge as used in the articles **[1,2,3]**: 

- **Karate Club**: a classic social network with 34 nodes and 78 edges.
- **Dolphins**: a social/biological network with 62 nodes and 159 edges.
- **Political Books**: a co-purchasing network of books about US politics.
- **College Football**: a schedule-based network of college football teams.
- **Jazz**: a collaboration network of jazz musicians.
- **C. elegans**: the neural network of the nematode worm.
- **E-mail**: an email interchange network from the University Rovira i Virgili.
- **PGP**: the giant component of the Pretty-Good-Privacy web of trust.
- **Condmat2003**: a condensed matter physics collaboration network.

Data source: David A. Bader, Henning Meyerhenke, Peter Sanders, Dorothea Wagner (eds.): Graph Partitioning and Graph Clustering. 10th DIMACS Implementation Challenge Workshop. February 13-14, 2012. Georgia Institute of Technology, Atlanta, GA. Contemporary Mathematics 588. American Mathematical Society and Center for Discrete Mathematics and Theoretical Computer Science, 2013.

In [1]:
using Graphs
using MatrixDepot
using SimpleWeightedGraphs
using Random

[ Info: verify download of index files...
[ Info: reading database
[ Info: adding metadata...
[ Info: adding svd data...
[ Info: writing database
[ Info: used remote sites are sparse.tamu.edu with MAT index and math.nist.gov with HTML index


In [2]:
karate_graph = smallgraph(:karate)
dolphins_graph = SimpleGraph(matrixdepot("Newman/dolphins"))
polbooks_graph = SimpleGraph(matrixdepot("Newman/polbooks"))
football_graph = SimpleGraph(matrixdepot("Newman/football"))
jazz_graph = SimpleGraph(matrixdepot("Arenas/jazz"))
celegans_graph = SimpleDiGraph(matrixdepot("Newman/celegansneural")) 
email_graph = SimpleDiGraph(matrixdepot("Arenas/email"))
pgp_graph = SimpleDiGraph(matrixdepot("Arenas/PGPgiantcompo"))
condmat_graph = SimpleDiGraph(matrixdepot("Newman/cond-mat-2003"))

datasets = Dict(
    "Karate Club" => karate_graph,
    "Dolphins" => dolphins_graph,
    "Political Books" => polbooks_graph,
    "College Football" => football_graph,
    "Jazz" => jazz_graph,
    "C. elegans" => celegans_graph,
    "E-mail" => email_graph,
    "PGP" => pgp_graph,
    "Condmat2003" => condmat_graph
)

println("Datasets loaded successfully:")
for (name, g) in datasets
    println("- ", name, ": ", nv(g), " nodes, ", ne(g), " edges")
end

Datasets loaded successfully:
- Karate Club: 34 nodes, 78 edges
- College Football: 115 nodes, 613 edges
- PGP: 10680 nodes, 48632 edges
- C. elegans: 297 nodes, 2345 edges
- Condmat2003: 31163 nodes, 240058 edges
- Political Books: 105 nodes, 441 edges
- Dolphins: 62 nodes, 159 edges
- Jazz: 198 nodes, 2742 edges
- E-mail: 1133 nodes, 10902 edges


# 2 - Heuristic methods

## 2.1 - Simple Label Propagation (LPA)

First, I implement a basic Label Propagation Algorithm ***[article 2, section 3.1]***. The procedure is intentionally simple:
1. every node is initially assigned a unique label;
2. at each iteration, each node adopts the most frequent label among its neighbors;
3. when there is a tie, one of the tied labels is chosen uniformly at random.

This version does not include modularity maximization or community merging, so the result can be unstable and may yield one or two communities depending on the random seed.

In [3]:
function lpa(g; max_iter=100, seed=1)
    rng = MersenneTwister(seed)
    n = nv(g)

    labels = collect(1:n)
    current_labels = copy(labels)

    for _ in 1:max_iter
        changed = false
        new_labels = copy(current_labels)

        for u in 1:n
            nbrs = [v for v in neighbors(g, u) if v != 0]
            if isempty(nbrs)
                continue
            end

            counts = Dict{Int, Int}()
            for v in nbrs
                lab = current_labels[v]
                counts[lab] = get(counts, lab, 0) + 1
            end

            max_count = maximum(values(counts))
            candidates = [lab for (lab, count) in counts if count == max_count]
            chosen_label = candidates[rand(rng, 1:length(candidates))]

            if new_labels[u] != chosen_label
                new_labels[u] = chosen_label
                changed = true
            end
        end

        current_labels = new_labels
        if !changed
            break
        end
    end

    return current_labels
end


lpa (generic function with 1 method)

## 2.2 - Label Propagation with Modularity Maximization (LPAm)

I now refine the label propagation result by using the modularity objective as a local optimization criterion.
Starting from the simple LPA partition, each node is revisited and moved to the neighboring community (or a new singleton community) that yields the highest modularity gain.
When several moves give the same modularity score, one of them is selected at random. ***[article 2, section 3.2]***

### 2.2.1 - Objective function 

To compare candidate partitions for modularity maximization, we compute Newman-Girvan modularity.
This score measures how much more edge weight lies inside communities than would be expected by chance in a random graph with the same node strengths.

The implementation below is generic:
- unweighted graphs are treated as if every existing edge has weight `1.0`
- weighted graphs built with `SimpleWeightedGraphs.jl` use their actual edge weights
- the community assignment is given as a label vector indexed by vertex

This lets us evaluate both `Graph` and `SimpleWeightedGraph` inputs using the same objective function.

In [4]:
function modularity(g, community)
    n = nv(g)
    @assert length(community) == n "community vector must have one label per vertex"

    has_weight = hasmethod(weight, Tuple{typeof(g), Int, Int})
    edge_weight(u, v) = has_weight ? weight(g, u, v) : 1.0

    m = 0.0
    degree_weight = zeros(Float64, n)
    for e in edges(g)
        u, v = src(e), dst(e)
        w = edge_weight(u, v)
        m += w
        degree_weight[u] += w
        degree_weight[v] += w
    end

    if m == 0.0
        return 0.0
    end

    two_m = 2.0 * m

    community_degree = Dict{Int, Float64}()
    for u in 1:n
        c = community[u]
        community_degree[c] = get(community_degree, c, 0.0) + degree_weight[u]
    end

    intra_weight = 0.0
    for e in edges(g)
        u, v = src(e), dst(e)
        if community[u] == community[v]
            intra_weight += edge_weight(u, v)
        end
    end

    Q = intra_weight / m
    for (_, Dt) in community_degree
        Q -= (Dt / two_m)^2
    end

    return Q
end

modularity (generic function with 1 method)

In [5]:
# Sanity check: modularity of the trivial (one-community)
# We should obtain 0 because we didn't compute the clusters of the network yet.
function whole_graph_modularity(g)
    trivial_community = ones(Int, nv(g))
    return modularity(g, trivial_community)
end

whole_graph_modularity (generic function with 1 method)

In [6]:
println("Dolphins whole-graph modularity: ", whole_graph_modularity(dolphins_graph))
println("Karate whole-graph modularity: ", whole_graph_modularity(karate_graph))

Dolphins whole-graph modularity: 0.0
Karate whole-graph modularity: 0.0


### 2.2.2 - LPAm Implementation

In [7]:
function initialize_label_state(g, initial_labels, degrees)
    n = nv(g)
    mapping = Dict{Int, Int}()
    next_id = 1
    for lab in unique(initial_labels)
        mapping[lab] = next_id
        next_id += 1
    end
    current_labels = [mapping[lab] for lab in initial_labels]

    max_possible_labels = n
    D = zeros(Int, max_possible_labels)
    nodes_in_label = [Set{Int}() for _ in 1:max_possible_labels]

    for u in 1:n
        lab = current_labels[u]
        push!(nodes_in_label[lab], u)
        D[lab] += degrees[u]
    end

    active_labels = Set(unique(current_labels))
    return current_labels, D, nodes_in_label, active_labels
end

function neighbor_label_counts(g, u, current_labels)
    label_counts = Dict{Int, Int}()
    for v in neighbors(g, u)
        if v != 0
            lab_v = current_labels[v]
            label_counts[lab_v] = get(label_counts, lab_v, 0) + 1
        end
    end
    return label_counts
end

function ensure_label_exists!(D, nodes_in_label)
    unused_label = findfirst(==(0), D)
    if unused_label === nothing
        push!(D, 0)
        push!(nodes_in_label, Set{Int}())
        unused_label = length(D)
    end
    return unused_label
end

function choose_best_label!(u, current_lab, D, nodes_in_label, degrees, two_m, label_counts, rng)
    best_score = -Inf
    best_label = current_lab

    for (cand, k_u_to_cand) in label_counts
        D_l = D[cand]
        score = k_u_to_cand - (degrees[u] * D_l) / two_m

        if score > best_score + 1e-12
            best_score = score
            best_label = cand
        elseif abs(score - best_score) <= 1e-12 && rand(rng) < 0.5
            best_label = cand
        end
    end

    unused_label = ensure_label_exists!(D, nodes_in_label)
    score_unused = 0.0 - (degrees[u] * D[unused_label]) / two_m

    if score_unused > best_score + 1e-12
        best_score = score_unused
        best_label = unused_label
    elseif abs(score_unused - best_score) <= 1e-12 && rand(rng) < 0.5
        best_label = unused_label
    end

    return best_label
end

function lpam(g; max_iter=20, seed=1)
    rng = MersenneTwister(seed)
    n = nv(g)
    two_m = 2.0 * ne(g)
    degrees = degree(g)

    initial_labels = lpa(g; seed=seed)
    current_labels, D, nodes_in_label, active_labels = initialize_label_state(g, initial_labels, degrees)

    iter = 0
    while !isempty(active_labels) && iter < max_iter
        iter += 1

        lab = pop!(active_labels)
        if isempty(nodes_in_label[lab])
            continue
        end

        nodes_to_check = collect(nodes_in_label[lab])
        shuffle!(rng, nodes_to_check)

        for u in nodes_to_check
            if current_labels[u] != lab
                continue
            end

            current_lab = current_labels[u]
            D[current_lab] -= degrees[u]

            label_counts = neighbor_label_counts(g, u, current_labels)
            best_label = choose_best_label!(u, current_lab, D, nodes_in_label, degrees, two_m, label_counts, rng)

            D[best_label] += degrees[u]

            if best_label != current_lab
                delete!(nodes_in_label[current_lab], u)
                push!(nodes_in_label[best_label], u)
                current_labels[u] = best_label

                push!(active_labels, current_lab)
                push!(active_labels, best_label)

                for v in neighbors(g, u)
                    push!(active_labels, current_labels[v])
                end
            end
        end
    end

    return current_labels
end


lpam (generic function with 1 method)

## 2.3 - Community Merging (LPAm+)



Compared with LPAm, LPAm+ adds a second refinement step in which neighboring communities are merged whenever that increases modularity ***[article 2, section 4]***. This helps avoid over-fragmentation and often yields a more stable partition than the local-move phase alone.

First, here are the helper functions.


In [8]:
function relabel_communities(labels)
    mapping = Dict{Int, Int}()
    next_id = 1
    relabeled = copy(labels)

    for i in eachindex(relabeled)
        lab = relabeled[i]
        if !haskey(mapping, lab)
            mapping[lab] = next_id
            next_id += 1
        end
        relabeled[i] = mapping[lab]
    end

    return relabeled
end

function communities_are_adjacent(g, labels, a, b)
    for u in 1:nv(g)
        if labels[u] != a
            continue
        end

        for v in neighbors(g, u)
            if v != 0 && labels[v] == b
                return true
            end
        end
    end

    return false
end

function merge_gain(g, labels, a, b)
    merged_labels = [lab == b ? a : lab for lab in labels]
    return modularity(g, merged_labels) - modularity(g, labels)
end

function best_merge_pair(g, labels)
    best_gain = 0.0
    best_pair = nothing

    communities = unique(labels)
    for i in 1:length(communities)-1
        for j in i+1:length(communities)
            a = communities[i]
            b = communities[j]

            if !communities_are_adjacent(g, labels, a, b)
                continue
            end

            gain = merge_gain(g, labels, a, b)
            if gain > best_gain + 1e-12
                best_gain = gain
                best_pair = (a, b)
            end
        end
    end

    return best_pair, best_gain
end

function merge_communities!(g, labels)
    current_labels = relabel_communities(labels)

    while true
        best_pair, best_gain = best_merge_pair(g, current_labels)
        if best_pair === nothing || best_gain <= 1e-12
            break
        end

        a, b = best_pair
        current_labels = [lab == b ? a : lab for lab in current_labels]
        current_labels = relabel_communities(current_labels)
    end

    return current_labels
end

function lpam_with_initial_labels(g, initial_labels; max_iter=20, seed=1)
    rng = MersenneTwister(seed)
    n = nv(g)
    two_m = 2.0 * ne(g)
    degrees = degree(g)

    current_labels, D, nodes_in_label, active_labels = initialize_label_state(g, initial_labels, degrees)

    iter = 0
    while !isempty(active_labels) && iter < max_iter
        iter += 1

        lab = pop!(active_labels)
        if isempty(nodes_in_label[lab])
            continue
        end

        nodes_to_check = collect(nodes_in_label[lab])
        shuffle!(rng, nodes_to_check)

        for u in nodes_to_check
            if current_labels[u] != lab
                continue
            end

            current_lab = current_labels[u]
            D[current_lab] -= degrees[u]

            label_counts = neighbor_label_counts(g, u, current_labels)
            best_label = choose_best_label!(u, current_lab, D, nodes_in_label, degrees, two_m, label_counts, rng)

            D[best_label] += degrees[u]

            if best_label != current_lab
                delete!(nodes_in_label[current_lab], u)
                push!(nodes_in_label[best_label], u)
                current_labels[u] = best_label

                push!(active_labels, current_lab)
                push!(active_labels, best_label)

                for v in neighbors(g, u)
                    push!(active_labels, current_labels[v])
                end
            end
        end
    end

    return current_labels
end

lpam_with_initial_labels (generic function with 1 method)

Then, here is the main LPAm+ function:

In [9]:
function lpam_plus(g; max_iter=20, seed=1)
    initial_labels = lpa(g; seed=seed)
    current_labels = lpam_with_initial_labels(g, initial_labels; max_iter=max_iter, seed=seed)

    while true
        n_communities_before = length(unique(current_labels))

        merged_labels = merge_communities!(g, current_labels)

        if length(unique(merged_labels)) == n_communities_before
            current_labels = merged_labels
            break
        end

        current_labels = lpam_with_initial_labels(g, merged_labels; max_iter=max_iter, seed=seed)
    end

    return current_labels
end

lpam_plus (generic function with 1 method)

# 3 - Experiments

I use the same following structure to print the results for each methods. 

In [10]:
datasets = [
    ("Karate", karate_graph),
    ("Dolphins", dolphins_graph),
    ("Political Books", polbooks_graph),
    ("College Football", football_graph), 
    ("Jazz", jazz_graph),
    ("C. elegans", celegans_graph),
    ("E-mail", email_graph),
    ("PGP", pgp_graph),
    ("Condmat2003", condmat_graph)
]

function print_performance_summary(method_name, method_fn)
    println("=== $(method_name) performance over all datasets ===")
    println("Dataset               | Communities | Modularity")
    println("----------------------+-------------+-----------")
    for (name, g) in datasets
        labels = method_fn(g)
        communities = length(unique(labels))
        q = modularity(g, labels)
        println(rpad(name, 22), "| ", lpad(communities, 11), " | ", round(q, digits=6))
    end
    println()
end


print_performance_summary (generic function with 1 method)

### 3.1 - LPA Performance 

In [11]:
print_performance_summary("LPA", g -> lpa(g; seed=42))

=== LPA performance over all datasets ===
Dataset               | Communities | Modularity
----------------------+-------------+-----------
Karate                |           2 | 0.021696
Dolphins              |           5 | 0.394624
Political Books       |           2 | 0.443732
College Football      |          10 | 0.592195
Jazz                  |           3 | 0.442774
C. elegans            |           3 | 0.011913
E-mail                |           3 | 0.01379
PGP                   |        1584 | 0.617823
Condmat2003           |        4127 | 0.641086



### 3.2 - LPAm Performance 

In [12]:
print_performance_summary("LPAm", g -> lpam(g; seed=42))

=== LPAm performance over all datasets ===
Dataset               | Communities | Modularity
----------------------+-------------+-----------
Karate                |           2 | 0.371795
Dolphins              |           4 | 0.526463
Political Books       |           2 | 0.443732
College Football      |          10 | 0.601328
Jazz                  |           3 | 0.444031
C. elegans            |           4 | 0.010751
E-mail                |         167 | 0.394615
PGP                   |        1575 | 0.619589
Condmat2003           |        4121 | 0.641286



print_performance_summary("LPAm", g -> lpam(g; seed=42))

### 3.3 - LPAm+ Performance 

In [ ]:
print_performance_summary("LPAm+", g -> lpam_plus(g; seed=42))

=== LPAm+ performance over all datasets ===
Dataset               | Communities | Modularity
----------------------+-------------+-----------
Karate                |           2 | 0.371795
Dolphins              |           4 | 0.526463
Political Books       |           2 | 0.443732
College Football      |           9 | 0.601499
Jazz                  |           3 | 0.444031
C. elegans            |           4 | 0.007552


# 4 - References

The references below provide the main sources used throughout this notebook.

**[1]** D. Aloise, G. Caporossi, P. Hansen, L. Liberti, S. Perron, and M. Ruiz, <i>Modularity maximization in networks by variable neighborhood search</i>, Contemporary Mathematics, vol. 588, pp. 113-127, 2013.  

**[2]** X. Liu and T. Murata, <i>Advanced modularity-specialized label propagation algorithm for detecting communities in networks</i>, Mar. 2010.

**[3]** P. Schuetz and A. Caflisch, <i>Efficient modularity optimization by multistep greedy algorithm and vertex mover refinement</i>, May 2008.